#### City Outing Forecast Prediction Tool
Purpose    : To give information about a city and predict the weather conditions so that a resident or traveller can go for outing \
LLM Method : We will use Anthropic "claude-sonnet-4-5" to take an input of city name, then provide details of top 5 attrractions, then call a tool to predict the weather on that day\
Input  : City Name and plan of travel with dates \
OutpUt : Top five attractions and current weather

This will help to understand how LLM can be used for calling tools externally to perform realworld actions

In [60]:
# Importing required Modules

import os
import json
from  dotenv import load_dotenv
from openai import OpenAI, api_key
import gradio as gr
import requests

In [61]:
# Importing required variables

load_dotenv(override=True)
openai_key = os.getenv("OPENAI_API_KEY")
weather_key = os.getenv("OPEN_WEATHER_API_KEY")

In [62]:
# Verifying the keys are loaded correctly

print(f"OpenAI Key       : {openai_key[:5]}")
print(f"Open Weather key : {weather_key[:5]}")

OpenAI Key       : sk-pr
Open Weather key : fd061


In [63]:
# initializing OpenAI SDK

openai=OpenAI()

In [64]:
# Defining function to collect weather details from OpenWeather API

def get_weather_data(latitude, longitude):
    """
    :param latitude: Latitude, decimal (-90; 90). If you need the geocoder to automatic convert city names and zip-codes to geo coordinates and the other way around,
    :param longitude: Longitude, decimal (-180; 180). If you need the geocoder to automatic convert city names and zip-codes to geo coordinates and the other way around,
    :return: weather outlook
    """
    weather_url = "https://api.openweathermap.org/data/2.5/weather"
    params = {
        "lat" :round(float(latitude), 2),
        "lon" :round(float(longitude), 2),
        "exclude": "minutely,hourly,daily,alerts",
        "appid" : weather_key,
        "units": "metric"
    }
    try:
        response = requests.get(weather_url, params=params)
        data = response.json()
        return data
    except Exception as error:
        print(f"Error : while connecting to OpenWeather API : {error}")

In [65]:
# Testing Weather function to make sure its working as expected

latitude = 32.77
longitude = 96.79
weather = get_weather_data(latitude, longitude)
weather
if weather:
    print(f"Current Temp: {weather['main']['temp']}°C")
    print(f"Weather: {weather['weather'][0]['description']}")
    print(f"Feels Like: {weather['main']['feels_like']}°C")
    print(f"Humidity: {weather['main']['humidity']}%")
    print(f"Visibility: {weather['visibility']} meters")

Current Temp: -7.45°C
Weather: clear sky
Feels Like: -7.45°C
Humidity: 49%
Visibility: 10000 meters


In [66]:
# Defining the tool for LLM to call and get weather data (This is the tool definition which we will provide to LLM and it will call this tool with appropriate arguments when it needs to get weather data)

get_weather_data_tool = {
    "name": "get_weather_data",
    "description": "Use this tool to get weather data for a specific latitude and longitude",
    "parameters": {
        "type": "object",
        "properties": {
            "latitude": {
                "type": "number",
                "description": "Latitude, decimal (-90; 90)."
            },
            "longitude": {
                "type": "number",
                "description": "Longitude, decimal (-180; 180)."
            }
        },
        "required": ["latitude", "longitude"],
        "additionalProperties": False
    }
}

In [55]:
# defining tool which will be used in LLM prompt

tools = [{"type": "function", "function": get_weather_data_tool }]

In [56]:
tools

[{'type': 'function',
  'function': {'name': 'get_weather_data',
   'description': 'Use this tool to get weather data for a specific latitude and longitude',
   'parameters': {'type': 'object',
    'properties': {'latitude': {'type': 'number',
      'description': 'Latitude, decimal (-90; 90).'},
     'longitude': {'type': 'number',
      'description': 'Longitude, decimal (-180; 180).'}},
    'required': ['latitude', 'longitude'],
    'additionalProperties': False}}}]

In [57]:
# Defining a system prompt to control the behaviour of the LLM

system_prompt = """You are a travel agent named Aparna trying to guide your customer to visit places in a city and its weather forecast.
                    You will need to ask your customer the city they are planning to travel.
                    Then need to find the find latitude and longitude coordinates for that city and call the get_weather_data tool with those coordinates.
                    When you send response back to user, make sure you are sending the information of weather forecast in a user friendly way, and not the raw data from the tool response.
                    You must also include top 5 attractions they can visit to.
                    Also best travel agent booking platforms available to book this trip which will save money for them
                    """

In [58]:
# Defining a chatbot function which will be used to interact with the user and also call the tool when needed

def chat(message, history):
    """
    Function to be used with in chatbot, this gives regular LLM response for normal queries,
    but when it needs to get weather data, it will call the tool and then use the response from tool to give final response to user.
    """
    messages = [{"role": "system", "content": system_prompt}] + history +  [{"role": "user", "content": message}]
    done = False
    while not done:
        model_name = "gpt-4o-mini"
        response = openai.chat.completions.create(
            model=model_name,
            messages=messages,
            tools=tools
        )
        message_obj = response.choices[0].message
        finish_reason = response.choices[0].finish_reason
        if finish_reason=="tool_calls":
            messages.append(message_obj)
            tool_calls = message_obj.tool_calls
            for tool_call in tool_calls:
                arguments = json.loads(tool_call.function.arguments)
                results = get_weather_data(**arguments)
                # Add tool response
                tool_response = {
                    "role": "tool",
                    "content": json.dumps(results),
                    "tool_call_id": tool_call.id
                }
                messages.append(tool_response)
        else:
            done = True
    return response.choices[0].message.content

In [59]:
# Initializing chat bot using gradio

gr.ChatInterface(chat).launch()


* Running on local URL:  http://127.0.0.1:7871
* To create a public link, set `share=True` in `launch()`.
